# Level 3 — Business Analytics

## Objective

This notebook answers business questions using SQL queries and summarizes insights for stakeholders.

In [1]:
import pandas as pd
import duckdb

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

con = duckdb.connect()

con.execute("""
CREATE OR REPLACE VIEW superstore_features AS
SELECT *
FROM read_csv_auto('../data/superstore_features.csv');
""")

con.execute("""
SELECT *
FROM superstore_features
LIMIT 5
""").df()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,fulfillment_days,profit_margin,order_year,order_month,customer_lifetime_sales,customer_span_days,customer_tier
0,2230,CA-2014-128055,2014-03-31,2014-04-05,Standard Class,AA-10315,Alex Avila,Consumer,United States,San Francisco,California,94122,West,OFF-BI-10004390,Office Supplies,Binders,GBC DocuBind 200 Manual Binding Machine,673.568,2,0.2,252.5880,5,0.3750,2014,3,5563.56,1186,Premium
1,2231,CA-2014-128055,2014-03-31,2014-04-05,Standard Class,AA-10315,Alex Avila,Consumer,United States,San Francisco,California,94122,West,OFF-AP-10002765,Office Supplies,Appliances,Fellowes Advanced Computer Series Surge Protec...,52.980,2,0.0,14.8344,5,0.2800,2014,3,5563.56,1186,Premium
2,5199,CA-2016-103982,2016-03-03,2016-03-08,Standard Class,AA-10315,Alex Avila,Consumer,United States,Round Rock,Texas,78664,Central,OFF-SU-10000151,Office Supplies,Supplies,High Speed Automatic Electric Letter Opener,3930.072,3,0.2,-786.0144,5,-0.2000,2016,3,5563.56,1186,Premium
3,5200,CA-2016-103982,2016-03-03,2016-03-08,Standard Class,AA-10315,Alex Avila,Consumer,United States,Round Rock,Texas,78664,Central,OFF-FA-10001332,Office Supplies,Fasteners,"Acco Banker's Clasps, 5 3/4""-Long",2.304,1,0.2,0.7776,5,0.3375,2016,3,5563.56,1186,Premium
4,5201,CA-2016-103982,2016-03-03,2016-03-08,Standard Class,AA-10315,Alex Avila,Consumer,United States,Round Rock,Texas,78664,Central,TEC-PH-10000895,Technology,Phones,Polycom VVX 310 VoIP phone,431.976,3,0.2,32.3982,5,0.0750,2016,3,5563.56,1186,Premium


## Selecting Variables for Business Analysis

The engineered dataset contains both the original retail variables and the features created in **Notebook 02**. While all variables remain available in the exported dataset, only those relevant to the business analyses in this notebook were selected.

The following columns were excluded for the reasons below:

- **`row_id`** was excluded because it serves only as a unique row identifier and does not provide meaningful information for the planned business analyses.

- **`order_date`** was excluded because the engineered features **`order_year`** and **`order_month`** provide the time granularity needed for this analysis.

- **`ship_date`** was excluded because the engineered feature **`fulfillment_days`** more directly measures shipping performance.

- **`customer_id`** was excluded in favor of **`customer_name`**, which produces more interpretable customer-level reports.

- **`product_id`** and **`product_name`** were excluded from product-level analysis because Notebook 01 identified inconsistencies in the relationship between these fields. Without sufficient information to determine the correct identifier-name relationships, using either field to make individual-product comparisons could produce unreliable conclusions. Product performance is therefore evaluated at the independently validated **`category`** and **`sub_category`** levels.

- **`country`** was excluded because every observation occurred in the United States, providing no additional analytical value.

- **`city`** and **`postal_code`** were excluded because they contain **531** and **631** unique values, respectively. For this analysis, **`region`** and **`state`** provide a more meaningful level of geographic aggregation while reducing unnecessary granularity.

The resulting dataset retains the variables most relevant to analyzing customer behavior, category and subcategory performance, geographic trends, profitability, and operational efficiency. This selection also ensures that the analyses rely only on fields whose relationships were validated during preprocessing, keeping the findings focused, interpretable, and defensible.



In [ ]:
con.execute("""
CREATE OR REPLACE VIEW analysis_data AS

SELECT
    -- Customer
    customer_name,
    segment,
    customer_lifetime_sales,  -- Engineered
    customer_span_days,       -- Engineered
    customer_tier,            -- Engineered

    -- Order and Time
    order_id,
    order_year,               -- Engineered
    order_month,              -- Engineered

    -- Geography
    region,
    state,

    -- Product Classification
    category,
    sub_category,

    -- Shipping
    ship_mode,
    fulfillment_days,         -- Engineered

    -- Sales and Profitability
    sales,
    quantity,
    discount,
    profit,
    profit_margin             -- Engineered

FROM superstore_features;
""")

con.execute("""
SELECT *
FROM analysis_data
LIMIT 5;
""").df()


,customer_name,segment,order_id,order_date,region,state,category,sub_category,ship_mode,sales,quantity,discount,profit,order_year,order_month,fulfillment_days,profit_margin,customer_lifetime_sales,customer_span_days,customer_tier
0,Alex Avila,Consumer,CA-2014-128055,2014-03-31,West,California,Office Supplies,Binders,Standard Class,673.568,2,0.2,252.5880,2014,3,5,0.3750,5563.56,1186,Premium
1,Alex Avila,Consumer,CA-2014-128055,2014-03-31,West,California,Office Supplies,Appliances,Standard Class,52.980,2,0.0,14.8344,2014,3,5,0.2800,5563.56,1186,Premium
2,Alex Avila,Consumer,CA-2016-103982,2016-03-03,Central,Texas,Office Supplies,Supplies,Standard Class,3930.072,3,0.2,-786.0144,2016,3,5,-0.2000,5563.56,1186,Premium
3,Alex Avila,Consumer,CA-2016-103982,2016-03-03,Central,Texas,Office Supplies,Fasteners,Standard Class,2.304,1,0.2,0.7776,2016,3,5,0.3375,5563.56,1186,Premium
4,Alex Avila,Consumer,CA-2016-103982,2016-03-03,Central,Texas,Technology,Phones,Standard Class,431.976,3,0.2,32.3982,2016,3,5,0.0750,5563.56,1186,Premium


## Customer Analysis

Customer analytics focuses on understanding purchasing behavior across individual customers and customer segments. By identifying high-value customers, comparing segment performance, and evaluating customer lifetime sales, we can better understand who generates the greatest value for the business and where customer retention efforts should be focused.

### Top Customers by Lifetime Sales

**Question:**

Who are the ten highest-value customers based on lifetime sales, and what do their ordering frequency, average spending per order, and observed customer span reveal about their purchasing behavior?

**Objective:**

Identify the ten customers with the highest cumulative sales and compare their total number of orders, average spending per order, and the number of days between their first and most recent recorded purchases. This analysis helps determine whether high customer value is driven primarily by frequent purchasing, larger orders, a longer purchasing history, or a combination of these factors.

**Variables:**

- `customer_name`
- `customer_lifetime_sales`
- `order_id`
- `sales`
- `customer_span_days`

In [3]:
con.execute("""
SELECT
    customer_name,
    ANY_VALUE(customer_lifetime_sales) AS customer_lifetime_sales,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(sales) / COUNT(DISTINCT order_id) AS avg_spend_per_order,
    ANY_VALUE(customer_span_days) AS customer_span_days,

    365.25 * SUM(sales) /
        NULLIF(ANY_VALUE(customer_span_days), 0)
        AS annualized_spending_rate

FROM analysis_data

GROUP BY customer_name

ORDER BY customer_lifetime_sales DESC

LIMIT 10;
""").df().style.hide(axis="index").format({
    "customer_lifetime_sales": "${:,.2f}",
    "total_orders": "{:,}",
    "avg_spend_per_order": "${:,.2f}",
    "customer_span_days": "{:,}",
    "annualized_spending_rate": "${:,.2f}"
})

customer_name,customer_lifetime_sales,total_orders,avg_spend_per_order,customer_span_days,annualized_spending_rate
Sean Miller,"$25,043.05",5,"$5,008.61","1,304","$7,014.55"
Tamara Chand,"$19,052.22",5,"$3,810.44",750,"$9,278.43"
Raymond Buch,"$15,117.34",6,"$2,519.56",542,"$10,187.47"
Tom Ashbrook,"$14,595.62",4,"$3,648.91","1,136","$4,692.83"
Adrian Barton,"$14,473.57",10,"$1,447.36","1,065","$4,963.82"
Ken Lonsdale,"$14,175.23",12,"$1,181.27","1,207","$4,289.56"
Sanjit Chand,"$14,142.33",9,"$1,571.37","1,068","$4,836.60"
Hunter Lopez,"$12,873.30",6,"$2,145.55","1,397","$3,365.76"
Sanjit Engle,"$12,209.44",11,"$1,109.95","1,350","$3,303.33"
Christopher Conant,"$12,129.07",5,"$2,425.81",543,"$8,158.64"


**Results:**

Sean Miller was the highest-value customer, generating **$25,043.05** across 5 orders, with an average spend of **$5,008.61 per order** and an observed purchasing span of **1,304 days**. Tamara Chand ranked second with **$19,052.22**, followed by Raymond Buch with **$15,117.34**.

Although Sean Miller generated the greatest lifetime sales, Raymond Buch had the highest observed annualized spending rate at **$10,187.47 per year**. Tamara Chand ranked second by this measure at **$9,278.43 per year**, followed by Christopher Conant at **$8,158.64 per year**.

**Key Insights:**

High customer value was produced through several different purchasing patterns. Sean Miller accumulated the most lifetime sales through relatively few, exceptionally large orders, averaging **$5,008.61 per order**. Ken Lonsdale followed a frequency-driven pattern, placing the most orders among the top ten but spending substantially less per order.

The annualized spending rate provides additional context by accounting for each customer's observed purchasing span. Raymond Buch accumulated sales at the fastest annualized rate despite ranking third in lifetime sales, while Hunter Lopez accumulated sales more slowly across the longest observed span. This demonstrates that lifetime sales alone can conceal meaningful differences in purchasing frequency, order size, and the pace at which customer value develops.


**Methodological Note:**

The annualized spending rate represents the rate at which recorded sales accumulated between a customer's first and most recent orders. It is a descriptive measure based on observed purchasing history and should not be interpreted as a forecast of future annual spending.

### Sales Performance by Customer Segment

**Question:**

How do revenue, profitability, customer count, and average sales per customer differ across customer segments?

**Objective:**

Compare total sales, total profit, profit margin, customer count, and average sales per customer across the Consumer, Corporate, and Home Office segments. This analysis identifies whether each segment's financial contribution is driven primarily by the size of its customer base, higher spending per customer, stronger profitability, or a combination of these factors.

**Variables:**

- `segment`
- `sales`
- `profit`
- `customer_id`

In [4]:
con.execute("""
SELECT
    segment,

    SUM(sales) AS total_sales,

    SUM(profit) AS total_profit,

    100.0 * SUM(sales) / SUM(SUM(sales)) OVER ()
        AS percentage_of_total_sales,

    100.0 * SUM(profit) / SUM(sales)
        AS profit_margin_percentage,

    COUNT(DISTINCT customer_name) AS customer_count,

    SUM(sales) / COUNT(DISTINCT customer_name)
        AS avg_sales_per_customer,


FROM analysis_data
GROUP BY segment
ORDER BY total_sales DESC;
""").df().style.hide(axis="index").format({
    "customer_count": "{:,}",
    "total_sales": "${:,.2f}",
    "percentage_of_total_sales": "{:.2f}%",
    "avg_sales_per_customer": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "profit_margin_percentage": "{:.2f}%"
})

segment,total_sales,total_profit,percentage_of_total_sales,profit_margin_percentage,customer_count,avg_sales_per_customer
Consumer,"$1,161,401.34","$134,119.21",50.56%,11.55%,409,"$2,839.61"
Corporate,"$706,146.37","$91,979.13",30.74%,13.03%,236,"$2,992.15"
Home Office,"$429,653.15","$60,298.68",18.70%,14.03%,148,"$2,903.06"


**Results:**

The Consumer segment generated the highest sales at **$1,161,401.34**, representing **50.56%** of total revenue. Corporate customers contributed **$706,146.37**, or **30.74%**, while the Home Office segment generated **$429,653.15**, accounting for the remaining **18.70%**.

**Key Insights:**

Consumer customers are the company’s largest source of revenue, producing slightly more than half of all sales. However, the Corporate segment also makes a substantial contribution, with Consumer and Corporate customers together accounting for **81.30%** of total sales. Although Home Office is the smallest segment, additional profitability analysis is needed before concluding that it is less valuable, since revenue alone does not account for costs or profit margins.

### Sales and Profitability by Customer Tier

**Question:**

How do overall financial contribution and average customer profitability differ across customer tiers?

**Objective:**

Compare customer count, total sales, total profit, average sales per customer, average profit per customer, and overall profit margin across customer tiers. This analysis evaluates whether higher-spending customer tiers also generate stronger profitability while accounting for differences in tier size.

**Variables:**

- `customer_tier`
- `customer_name`
- `sales`
- `profit`

In [5]:
con.execute("""
SELECT
    customer_tier,
    COUNT(DISTINCT customer_name) AS customer_count,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,

    SUM(sales) / COUNT(DISTINCT customer_name)
        AS avg_sales_per_customer,

    SUM(profit) / COUNT(DISTINCT customer_name)
        AS avg_profit_per_customer,

    100.0 * SUM(profit) / SUM(sales)
        AS profit_margin_percentage

FROM analysis_data

GROUP BY customer_tier

ORDER BY avg_sales_per_customer DESC;
""").df().style.hide(axis="index").format({
    "customer_count": "{:,}",
    "total_sales": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "avg_sales_per_customer": "${:,.2f}",
    "avg_profit_per_customer": "${:,.2f}",
    "profit_margin_percentage": "{:.2f}%"
})

customer_tier,customer_count,total_sales,total_profit,avg_sales_per_customer,avg_profit_per_customer,profit_margin_percentage
Elite,80,"$709,342.71","$122,167.79","$8,866.78","$1,527.10",17.22%
Premium,119,"$557,107.99","$65,687.41","$4,681.58",$552.00,11.79%
High Value,198,"$574,885.76","$52,925.88","$2,903.46",$267.30,9.21%
Standard,396,"$455,864.40","$45,615.94","$1,151.17",$115.19,10.01%


**Results:**

The Elite tier contained only **80 customers** but generated the highest total sales at **$709,342.71** and the highest total profit at **$122,167.79**. It also led all tiers in average sales per customer at **$8,866.78**, average profit per customer at **$1,527.10**, and overall profit margin at **17.22%**.

Premium customers averaged **$4,681.58** in sales and **$552.00** in profit per customer. Although the High Value tier generated slightly more total sales than Premium—**$574,885.76** compared with **$557,107.99**—it contained 79 more customers and produced lower average sales, average profit, and profit margin. The Standard tier contained the most customers at **396**, but had the lowest average sales and profit per customer.

**Key Insights:**

Elite customers are disproportionately valuable. Despite representing approximately **10% of the customer base**, they contributed about **30.88% of total sales** and **42.66% of total profit**. Their strong performance across both per-customer measures and profit margin indicates that their value is not simply caused by the tier's size.

The comparison between Premium and High Value customers demonstrates why totals should be interpreted alongside customer counts. High Value generated slightly more total sales because it contained more customers, while the average Premium customer generated substantially greater sales and more than twice as much profit. The High Value tier also had the lowest profit margin at **9.21%**, suggesting that discounting, product mix, or other cost drivers may warrant further investigation. Retention efforts should prioritize Elite and Premium customers while profitability improvement efforts may be especially valuable within the High Value tier.

## Geographic Analysis

Geographic analysis evaluates business performance across regions and states. Comparing sales, profitability, and customer activity by location helps identify high-performing markets, uncover regional trends, and highlight areas that may benefit from targeted business strategies.

### Financial Performance by Region

**Question:**

Which regions generate the greatest financial value when considering sales, profit, profit margin, and average order value?

**Objective:**

Compare total orders, sales, profit, profit margin, and average order value across regions. This analysis determines whether regional performance is driven by greater order volume, higher-value orders, stronger profitability, or a combination of these factors.

**Variables:**

- `region`
- `order_id`
- `sales`
- `profit`

In [ ]:
con.execute("""
SELECT
    region,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,

    100.0 * SUM(sales) / SUM(SUM(sales)) OVER ()
        AS percentage_of_total_sales,

    SUM(sales) / COUNT(DISTINCT order_id)
        AS avg_order_value,

    100.0 * SUM(profit) / SUM(sales)
        AS profit_margin_percentage

FROM analysis_data

GROUP BY region

ORDER BY total_sales DESC;
""").df().style.hide(axis="index").format({
    "total_orders": "{:,}",
    "total_sales": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "percentage_of_total_sales": "{:.2f}%",
    "avg_order_value": "${:,.2f}",
    "profit_margin_percentage": "{:.2f}%"
})

region,total_orders,total_sales,total_profit,percentage_of_total_sales,avg_order_value,profit_margin_percentage
West,"1,611","$725,457.82","$108,418.45",31.58%,$450.32,14.94%
East,"1,401","$678,781.24","$91,522.78",29.55%,$484.50,13.48%
Central,"1,175","$501,239.89","$39,706.36",21.82%,$426.59,7.92%
South,822,"$391,721.91","$46,749.43",17.05%,$476.55,11.93%


**Results:**

The West was the strongest-performing region, generating **$725,457.82** in sales and **$108,418.45** in profit across 1,611 orders. It contributed **31.58%** of total sales and achieved the highest regional profit margin at **14.94%**.

The East ranked second in total sales at **$678,781.24** and had the highest average order value at **$484.50**. The Central region generated **$501,239.89** in sales but produced only **$39,706.36** in profit, resulting in the lowest regional profit margin at **7.92%**. The South generated the lowest total sales but had a relatively strong average order value of **$476.55**.

**Key Insights:**

The West's leading performance was supported by a combination of the greatest order volume, highest total sales, highest total profit, and strongest profit margin. The East also performed strongly, generating the highest average order value and the second-highest profit margin.

The Central region warrants further investigation because its **7.92% profit margin** was substantially below the other regions despite generating more than half a million dollars in sales. This suggests that revenue volume alone did not translate into equally strong financial returns. In contrast, the South's relatively high average order value and **11.93% margin** indicate that its lower total contribution was driven more by limited order volume than weak order economics.

### Highest-Performing States

**Question:**

Which ten states generate the highest total sales, and how do their profitability and average order values compare?

**Objective:**

Identify the ten states with the highest sales and compare their total orders, total profit, average order value, and profit margin. This analysis determines whether the largest state markets also generate strong financial returns.

**Variables:**

- `state`
- `region`
- `order_id`
- `sales`
- `profit`

In [11]:
con.execute("""
SELECT
    state,
    region,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,

    SUM(sales) / COUNT(DISTINCT order_id)
        AS avg_order_value,

    100.0 * SUM(profit) / SUM(sales)
        AS profit_margin_percentage

FROM analysis_data

GROUP BY state, region

ORDER BY total_sales DESC

LIMIT 10;
""").df().style.hide(axis="index").format({
    "total_orders": "{:,}",
    "total_sales": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "avg_order_value": "${:,.2f}",
    "profit_margin_percentage": "{:.2f}%"
})

state,region,total_orders,total_sales,total_profit,avg_order_value,profit_margin_percentage
California,West,"1,021","$457,687.63","$76,381.39",$448.27,16.69%
New York,East,562,"$310,876.27","$74,038.55",$553.16,23.82%
Texas,Central,487,"$170,188.05","$-25,729.36",$349.46,-15.12%
Washington,West,256,"$138,641.27","$33,402.65",$541.57,24.09%
Pennsylvania,East,288,"$116,511.91","$-15,559.96",$404.56,-13.35%
Florida,South,200,"$89,473.71","$-3,399.30",$447.37,-3.80%
Illinois,Central,276,"$80,166.10","$-12,607.89",$290.46,-15.73%
Ohio,East,236,"$78,258.14","$-16,971.38",$331.60,-21.69%
Michigan,Central,117,"$76,269.61","$24,463.19",$651.88,32.07%
Virginia,South,115,"$70,636.72","$18,597.95",$614.23,26.33%


**Results:**

California generated the highest state-level sales at **$457,687.63** and the highest total profit at **$76,381.39**. New York ranked second in sales at **$310,876.27** and produced nearly as much profit as California—**$74,038.55**—despite recording 459 fewer orders. New York's profit margin was **23.82%**, compared with California's **16.69%**.

Several high-revenue states were not profitable. Texas ranked third in sales at **$170,188.05** but generated a **$25,729.36 loss**. Pennsylvania, Florida, Illinois, and Ohio also appeared among the ten highest-sales states while producing negative total profit. Michigan had the highest average order value at **$651.88** and the strongest profit margin at **32.07%** among the ten states shown.

**Key Insights:**

High sales did not consistently indicate strong financial performance. California and New York combined substantial revenue with positive profit, while Texas generated the third-highest sales but the largest state-level loss. Five of the ten highest-sales states produced negative profit, demonstrating that market size should not be evaluated independently of profitability.

New York was particularly efficient, producing nearly as much total profit as California from substantially fewer orders. Michigan and Virginia also produced strong average order values and profit margins despite their lower sales totals. These states may represent financially efficient markets with potential for carefully targeted growth, while Texas, Ohio, Pennsylvania, Illinois, and Florida require investigation into the factors reducing profitability.

### States Generating Financial Losses

**Question:**

Which states generated an overall financial loss despite producing sales?

**Objective:**

Identify states with negative total profit and compare their sales, order volume, average line-level discount, and profit margin. This analysis highlights geographic markets where revenue is not translating into positive financial returns and provides an initial indication of whether discounting may warrant further investigation.

**Variables:**

- `state`
- `region`
- `order_id`
- `sales`
- `profit`
- `discount`

In [12]:
con.execute("""
SELECT
    state,
    region,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,

    100.0 * AVG(discount)
        AS avg_line_discount_percentage,

    100.0 * SUM(profit) / SUM(sales)
        AS profit_margin_percentage

FROM analysis_data

GROUP BY state, region

HAVING SUM(profit) < 0

ORDER BY total_profit;
""").df().style.hide(axis="index").format({
    "total_orders": "{:,}",
    "total_sales": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "avg_line_discount_percentage": "{:.2f}%",
    "profit_margin_percentage": "{:.2f}%"
})

state,region,total_orders,total_sales,total_profit,avg_line_discount_percentage,profit_margin_percentage
Texas,Central,487,"$170,188.05","$-25,729.36",37.02%,-15.12%
Ohio,East,236,"$78,258.14","$-16,971.38",32.49%,-21.69%
Pennsylvania,East,288,"$116,511.91","$-15,559.96",32.86%,-13.35%
Illinois,Central,276,"$80,166.10","$-12,607.89",39.00%,-15.73%
North Carolina,South,136,"$55,603.16","$-7,490.91",28.35%,-13.47%
Colorado,West,79,"$32,108.12","$-6,527.86",31.65%,-20.33%
Tennessee,South,91,"$30,661.87","$-5,341.69",29.13%,-17.42%
Arizona,West,108,"$35,282.00","$-3,427.92",30.36%,-9.72%
Florida,South,200,"$89,473.71","$-3,399.30",29.93%,-3.80%
Oregon,West,56,"$17,431.15","$-1,190.47",28.87%,-6.83%


**Results:**

Ten states generated an overall financial loss. Texas produced the largest loss at **$25,729.36**, despite generating **$170,188.05** in sales across 487 orders. Ohio had the most negative profit margin at **-21.69%**, followed by Colorado at **-20.33%** and Tennessee at **-17.42%**.

Illinois recorded the highest average line-level discount among the loss-generating states at **39.00%**, while Texas averaged **37.02%**. Across all ten unprofitable states, average line-level discounts ranged from **28.35% to 39.00%**.

**Key Insights:**

The loss-generating states produced meaningful revenue, but their sales did not translate into positive financial returns. Texas is the most significant concern because it combined the highest sales and order volume among the unprofitable states with the largest total loss. Ohio and Colorado also warrant attention because more than 20% of their sales revenue was lost based on their overall profit margins.

The consistently elevated average discounts among these states suggest that discounting may be associated with weak geographic profitability. However, this table alone does not establish that discounts caused the losses because it does not compare discount patterns with profitable states or control for product mix. The relationship between discounting and profitability should therefore be evaluated directly in the Sales and Profitability Analysis section.

## Product Classification Analysis

Product analysis examines sales and profitability across product categories, subcategories, and individual products. The objective is to identify top-performing products, recognize underperforming inventory, and understand which areas of the product portfolio contribute most to overall business success.

## Sales & Profitability Analysis

Sales and profitability analysis investigates the financial performance of the business by examining revenue, discounts, profit, and profit margins. This section explores how pricing and discounting strategies influence profitability and identifies opportunities to improve financial performance.

## Shipping & Time Analysis

Shipping and time analysis evaluates operational efficiency and temporal business trends. By analyzing fulfillment times, shipping methods, and sales performance across months and years, this section identifies seasonal patterns, monitors delivery performance, and uncovers trends that can support operational planning.

In [6]:
con.sql("""
SELECT
    order_year,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,
    SUM(sales) / COUNT(DISTINCT order_id) AS avg_order_value,
    100.0 * SUM(profit) / SUM(sales) AS profit_margin_percentage
FROM superstore_features
GROUP BY order_year
ORDER BY order_year;
""").df().style.hide(axis="index").format({
    "total_sales": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "avg_order_value": "${:,.2f}",
    "profit_margin_percentage": "{:.2f}%"
})

order_year,total_orders,total_sales,total_profit,avg_order_value,profit_margin_percentage
2014,969,"$484,247.50","$49,543.97",$499.74,10.23%
2015,1038,"$470,532.51","$61,618.60",$453.31,13.10%
2016,1315,"$609,205.60","$81,795.17",$463.27,13.43%
2017,1687,"$733,215.26","$93,439.27",$434.63,12.74%


In [7]:
con.execute('''
SELECT
    segment,
    ROUND(AVG(fulfillment_days),2) AS avg_fulfillment_days,
    MIN(fulfillment_days) AS min_days,
    MEDIAN(fulfillment_days) AS median_days,
    MAX(fulfillment_days) AS max_days
FROM superstore_features
GROUP BY segment
ORDER BY avg_fulfillment_days;
''').df()

,segment,avg_fulfillment_days,min_days,median_days,max_days
0,Home Office,3.92,0,4.0,7
1,Consumer,3.94,0,4.0,7
2,Corporate,4.01,0,4.0,7


In [8]:
con.execute("""
SELECT
    ship_mode,
    ROUND(AVG(fulfillment_days), 2) AS avg_fulfillment_days,
    MIN(fulfillment_days) AS min_days,
    MAX(fulfillment_days) AS max_days
FROM superstore_features
GROUP BY ship_mode
ORDER BY avg_fulfillment_days;
""").df()

,ship_mode,avg_fulfillment_days,min_days,max_days
0,Same Day,0.04,0,1
1,First Class,2.18,1,4
2,Second Class,3.24,1,5
3,Standard Class,5.01,3,7
